## Preprocessing
**Umfang:** 
* Konvertieren der Dateien, Extrahieren des Texts, Extrahieren der Metadaten
* Nachbearbeiten des Texts: Entfernen wiederkehrender Textbausteine wie "Follow us on social media"
* Entfernen exakter Duplikate
* Entfernen von Ähnlichen Texten
  
**Folgende Pakete werden verwendet:**

In [1]:
#import os
#import csv
#from striprtf.striprtf import rtf_to_text
#import re
#import hashlib
#from datasketch import MinHash, MinHashLSH
#import string
#import pandas as pd
#import shutil

### Einlesen der RTF-Dateien, Extrahieren der Metadaten

In [2]:
import os
import csv
from striprtf.striprtf import rtf_to_text

# Eingabe-/Ausgabe-Pfade
input_folder = '/Users/lennertbusse/Documents/01Universität/DH/Projekt/texte_rtf' 
output_csv = 'artikel_uebersicht.csv'  
text_output_folder = '/Users/lennertbusse/Documents/01Universität/DH/Projekt/texte_txt'

# Felder für CSV
csv_fields = [
    'ID', 'Titel', 'Zeitung', 'Datum', 'Section', 'Byline',
    'Subject', 'Industry', 'Person', 'Geographic', 'Load-Date', 'Body-TXT-Datei'
]

def extract_metadata_and_body(text):
    lines = text.splitlines()

    def get_field(label):
        for line in lines:
            if line.startswith(label + ':'):
                return line.split(':', 1)[-1].strip()
        return ''

    titel = ''
    datum = ''

    try:
        ny_index = next(i for i, line in enumerate(lines) if "The New York Times" in line)
        potential_title_lines = lines[:ny_index]
        titel = ' '.join([line.strip() for line in potential_title_lines if line.strip()])
        datum = lines[ny_index + 1].strip() if len(lines) > ny_index + 1 else ''
    except StopIteration:
        pass

    # Body extrahieren
    body_lines = []
    in_body = False
    for line in lines:
        if line.strip() == 'Body':
            in_body = True
            continue
        if in_body:
            if line.strip().startswith(('https://', 'http://')):
                break
            if line.strip().startswith(('PHOTO', 'PHOTOS')):
                break
            if line.strip() in ["Graphic"]:
                break
            if line.strip() in ["Classification"]:
                break
            body_lines.append(line)

    body = '\n'.join(body_lines).strip()

    metadata = {
        'Titel': titel,
        'Zeitung': 'The New York Times',
        'Datum': datum,
        'Section': get_field('Section'),
        'Byline': get_field('Byline'),
        'Subject': get_field('Subject'),
        'Industry': get_field('Industry'),
        'Person': get_field('Person'),
        'Geographic': get_field('Geographic'),
        'Load-Date': get_field('Load-Date'),
        'Body': body
    }

    return metadata

os.makedirs(text_output_folder, exist_ok=True)

def main():
    counter = 0  # Start-ID

    with open(output_csv, mode='w', newline='', encoding='utf-8') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=csv_fields, delimiter=';', quoting=csv.QUOTE_MINIMAL)
        writer.writeheader()

        for filename in sorted(os.listdir(input_folder)):
            if filename.lower().endswith('.rtf'):
                if counter > 14000:
                    print("Maximale Artikelanzahl (14000) erreicht.")
                    break

                counter += 1
                doc_id = f'{counter:05d}'  # 5-stellige Zahl mit führenden Nullen

                path = os.path.join(input_folder, filename)
                with open(path, 'r', encoding='utf-8') as file:
                    rtf = file.read()
                    text = rtf_to_text(rtf)
                    data = extract_metadata_and_body(text)

                    txt_filename = f'{doc_id}.txt'
                    txt_path = os.path.join(text_output_folder, txt_filename)

                    # Titel und Body in die Textdatei schreiben
                    with open(txt_path, 'w', encoding='utf-8') as txt_file:
                        txt_file.write(f"{data['Titel']}\n\n{data['Body']}")

                    row = {
                        'ID': doc_id,
                        'Titel': data['Titel'],
                        'Zeitung': data['Zeitung'],
                        'Datum': data['Datum'],
                        'Section': data['Section'],
                        'Byline': data['Byline'],
                        'Subject': data['Subject'],
                        'Industry': data['Industry'],
                        'Person': data['Person'],
                        'Geographic': data['Geographic'],
                        'Load-Date': data['Load-Date'],
                        'Body-TXT-Datei': txt_filename
                    }

                    writer.writerow(row)
                    print(f"{filename} → {txt_filename}")

if __name__ == '__main__':
    main()

$100 Billion In the Hands Of a Computer.RTF → 00001.txt
$2 MILLION DAMAGE SUIT BY LILCO CHALLENGES OPPONENTS OF SHOREAM A-PLANT.RTF → 00002.txt
$20 BILLION VOTED FOR NUCLEAR FUSION.RTF → 00003.txt
$25 MILLION GIVEN FOR STUDIES TO PREVENT ATOM WAR.RTF → 00004.txt
$36 Billion Worth of Secrets.RTF → 00005.txt
$37 MILLION WON BY ATOMIC PLANT AT 3 MILE ISLAND.RTF → 00006.txt
$50 MILLION QUESTION.RTF → 00007.txt
$50 Million Offer Aims at Curbing Efforts to Make Nuclear Fuel.RTF → 00008.txt
$59 Million, Gone_ How Bikini Atoll Leaders Blew Through U.S. Trust Fund.RTF → 00009.txt
'25 Wars Are Still Going On'.RTF → 00010.txt
'50s Nuclear Target List Offers Chilling Insight.RTF → 00011.txt
'AN INTERESTING PHYSICS PROBLEM'.RTF → 00012.txt
'BIG MO' HAUNTED BY HISTORIC WORDS.RTF → 00013.txt
'Barbie' Powers Past Box Office Predictions.RTF → 00014.txt
'COSMIC STRINGS' IN DAWN OF GALAXIES EXPLORED.RTF → 00015.txt
'Cold Fusion' Claimants Review Puzzling Results.RTF → 00016.txt
'Cold Fusion' Patents Soug

### Einlesen der konvertierten Dateien

In [3]:
import os

ordner_pfad = "/Users/lennertbusse/Documents/01Universität/DH/Projekt/texte_txt" #Pfad zum Ergebniss aus vorherigem Schritt

texts = []
filenames = []

# Alle .txt-Dateien sortiert einlesen
for dateiname in sorted(os.listdir(ordner_pfad)):
    if dateiname.endswith(".txt"):
        dateipfad = os.path.join(ordner_pfad, dateiname)
        with open(dateipfad, "r", encoding="utf-8") as f:
            inhalt = f.read().strip()
            if inhalt:
                texts.append(inhalt)
                filenames.append(dateiname)

print(f"{len(texts)} Texte eingelesen.")

13625 Texte eingelesen.


### Nachbearbeiten der Texte

In [4]:
import os
import re

# Refined regex: match "Follow ..." only at start of a line (optional leading whitespace), case insensitive
follow_pattern = re.compile(
    r"^\s*Follow\s+(us|me|the\s+New\s+York\s+Times)",
    re.IGNORECASE | re.MULTILINE
)

footer_pattern = re.compile(
    r"""(
        ^\s*The\s+Times\s+is\s+committed\s+to\s+publishing.*?$|
        ^\s*[\w\s]+?\s+is\s+a\s+reporter\s+for\s+The\s+Times.*?$|
        ^\s*[\w\s]+?\s+contributed\s+reporting.*?$
    )""",
    re.IGNORECASE | re.MULTILINE | re.VERBOSE
)

affected_files = []
scanned_files = 0

for root, dirs, files in os.walk(ordner_pfad):
    for file_name in files:
        if file_name.endswith(".txt"):
            scanned_files += 1
            file_path = os.path.join(root, file_name)
            
            with open(file_path, 'r', encoding='utf-8') as f:
                content = f.read()

            # Erst nach "Follow…" suchen
            match = follow_pattern.search(content)
            if match:
                content = content[:match.start()].rstrip() + '\n'

            # Dann Footer-Sätze entfernen (kann mehrere sein!)
            content_cleaned = footer_pattern.sub('', content).strip()

            if content != content_cleaned:
                affected_files.append(os.path.relpath(file_path, ordner_pfad))
                with open(file_path, 'w', encoding='utf-8') as f:
                    f.write(content_cleaned)

# Ergebnis ausgeben
if affected_files:
    print(f"{len(affected_files)} Dateien bereinigt.")
    for f in affected_files:
        print(f)
else:
    print("Keine Dateien mit Footer-Phrasen erkannt.")

573 Dateien bereinigt.
02419.txt
13272.txt
13266.txt
01879.txt
06858.txt
05389.txt
11115.txt
11129.txt
13058.txt
11840.txt
06482.txt
11854.txt
04336.txt
02747.txt
06290.txt
05941.txt
02948.txt
01058.txt
09376.txt
01919.txt
08097.txt
01925.txt
01931.txt
00391.txt
01930.txt
00390.txt
01924.txt
01918.txt
01059.txt
05571.txt
13313.txt
04492.txt
05940.txt
06291.txt
02746.txt
06252.txt
00623.txt
11060.txt
11841.txt
11128.txt
13059.txt
03538.txt
11114.txt
11100.txt
05175.txt
13298.txt
05388.txt
06859.txt
01878.txt
06871.txt
13267.txt
02418.txt
05377.txt
13273.txt
13265.txt
08889.txt
08686.txt
04533.txt
11857.txt
06481.txt
06278.txt
11062.txt
03466.txt
13107.txt
04335.txt
02977.txt
09375.txt
00392.txt
01932.txt
06085.txt
01926.txt
06091.txt
06090.txt
01927.txt
02584.txt
01933.txt
12998.txt
05572.txt
02976.txt
06279.txt
03467.txt
11842.txt
05823.txt
04532.txt
11498.txt
04917.txt
06872.txt
08687.txt
13258.txt
08888.txt
08122.txt
06137.txt
13248.txt
11477.txt
00585.txt
00022.txt
12354.txt
11852.t

### Exakte Duplikate 

Der Inhalt der Dateien muss identisch sein, damit sie mit dieser Methode gefiltert werden.

In [5]:
import hashlib
import csv

def hash_text(text):
    return hashlib.md5(text.strip().encode('utf-8')).hexdigest()

# Duplikate entfernen 
# dafür sorgen dass dateinamen weiterhin richtig zugeordent werden

unique_texts_dict = {}
unique_texts = []
unique_filenames = []

for text, filename in zip(texts, filenames):
    h = hash_text(text)
    if h not in unique_texts_dict:
        unique_texts_dict[h] = text
        unique_texts.append(text)
        unique_filenames.append(filename)

unique_texts = list(unique_texts_dict.values())
print(f"Reduziert von {len(texts)} auf {len(unique_texts)} Texte durch exaktes Filtern.")

Reduziert von 13625 auf 11742 Texte durch exaktes Filtern.


### Nahe Duplikate I

**Wahrscheinliche Ähnlichkeit**

Die Inhalte der Dateien müssen nicht identisch sein, damit Dateien herausgefiltert werden. Es reicht, wenn sie sehr ähnlich sind. 
Mit MinHash wird die wahrscheinliche Ähnlichkeit berechnet. 

In [6]:
%%time
#pip install datasketch #ggf. installieren 

from datasketch import MinHash, MinHashLSH
import csv

def get_minhash(text, num_perm=256): #Anzahl der Hash-Funktionen – je höher, desto genauer (Standard: 128 oder 256)
    m = MinHash(num_perm=num_perm)
    for token in text.split():  # oder .lower().split() für robustere Ähnlichkeit
        m.update(token.encode('utf8'))
    return m

# LSH initialisieren
lsh = MinHashLSH(threshold=0.60, num_perm=256)
minhashes = {}

# MinHashes berechnen und einfügen
for i, text in enumerate(unique_texts):
    fname = unique_filenames[i]
    mh = get_minhash(text)
    lsh.insert(fname, mh)
    minhashes[fname] = mh

# Near-Duplicates finden
near_duplicates = set()

for i in range(len(unique_texts)):
    fname_i = unique_filenames[i]
    result = lsh.query(minhashes[fname_i])
    for fname_j in result:
        if fname_i != fname_j:
            pair = tuple(sorted((fname_i, fname_j)))
            near_duplicates.add(pair)

print(f"Es wurden {len(near_duplicates)} wahrscheinliche Near-Duplicate-Paare gefunden")

Es wurden 861 wahrscheinliche Near-Duplicate-Paare gefunden
CPU times: user 6min 2s, sys: 3.69 s, total: 6min 5s
Wall time: 6min 22s


### Nahe Duplikate II

Es wird die Jaccard-Ähnlichkeit der wahrscheinlich ähnlichen Paare berechnet. 
Anschließend wird eine CSV-Datei gespechert, die zur Überprüfung heranzezogen werden kann

In [7]:
duplicate_pairs = []

for a, b in near_duplicates:
        jsim = minhashes[a].jaccard(minhashes[b]) #jsim = jaccard similarity
        duplicate_pairs.append((a, b, jsim))

duplicate_pairs_sorted = sorted(duplicate_pairs, key=lambda x: x[2], reverse=True) #sortieren

with open("Dopplungen.csv", "w", newline="", encoding="utf-8") as file: 
    writer = csv.writer(file)
    writer.writerow(["Datei 1", "Datei 2", "Jaccard-Ähnlichkeit"])
    writer.writerows(duplicate_pairs_sorted)

print(f"Es wurde die Jaccard-Ähnlichkeit von {len(duplicate_pairs)} wahrscheinlichen Near Duplicates berrechnet und in 'Dopplungen.csv' gespeichert.")
for row in duplicate_pairs_sorted[:3]: 
    print(f"{row[0]} <-> {row[1]} | Jaccard-Ähnlichkeit = {row[2]:.4f}")

Es wurde die Jaccard-Ähnlichkeit von 861 wahrscheinlichen Near Duplicates berrechnet und in 'Dopplungen.csv' gespeichert.
03103.txt <-> 03104.txt | Jaccard-Ähnlichkeit = 1.0000
09826.txt <-> 09827.txt | Jaccard-Ähnlichkeit = 1.0000
00665.txt <-> 00667.txt | Jaccard-Ähnlichkeit = 0.9961


### Dopplungen entfernen

Für alle Dateien aus Paare mit einer Jaccard-Ähnlichkeit von größer gleich 0,7 soll die Menge der Zeichen gezählt werden. Die Datei mit weniger Zeichen soll entfernt werden.

In [8]:
import string 

def text_length_clean(text): #ohne Satzzeichen
    text_clean = text.translate(str.maketrans('','', string.punctuation))
    return len(text_clean)

filename_to_text = dict(zip(unique_filenames, unique_texts)) #mapping

files_to_remove = set() #identifikation der Paare

for a, b, jsim in duplicate_pairs_sorted:
    if jsim >= 0.7:
        if a in filename_to_text and b in filename_to_text:
            len_a = text_length_clean(filename_to_text[a]) if a in filename_to_text else "?"
            len_b = text_length_clean(filename_to_text[b]) if b in filename_to_text else "?"
            if len_a < len_b:
                files_to_remove.add(a)
            else:
                files_to_remove.add(b)

filtered_texts = []
filtered_filenames = []

for fname, text in zip(unique_filenames, unique_texts):
    if fname not in files_to_remove:
        filtered_filenames.append(fname)
        filtered_texts.append(text)

unique_texts = filtered_texts
unique_filenames = filtered_filenames

print(f"{len(files_to_remove)} Near Duplicates wurden entfernt.") 

804 Near Duplicates wurden entfernt.


### Texte in CSV speichern für Sentimentanalyse und Visualisierungen

In [9]:
import pandas as pd

filename_to_text = dict(zip(filtered_filenames, filtered_texts))

df = pd.read_csv("artikel_uebersicht.csv", sep=";", encoding="utf-8", quotechar='"')

df["Text"] = df["Body-TXT-Datei"].apply(lambda fname: filename_to_text.get(fname, None))

df.to_csv("artikel_uebersicht_text.csv", sep=";", index=False, encoding="utf-8", quotechar='"')

print("Die Artikeltexte wurden der CSV zugeordnet und gespeichert.")

Die Artikeltexte wurden der CSV zugeordnet und gespeichert.


### Datum vereinheitlichen 

In [10]:
import pandas as pd
from dateutil import parser

df = pd.read_csv("artikel_uebersicht_text.csv", sep = ";", encoding = "utf-8", quotechar = '"')
def parse_date(datum, load_date, text):
    for value in [datum, load_date, text]:
        try:
            if pd.notna(value):
                parsed = parser.parse(value, fuzzy=True)
                return parsed.strftime("%Y-%m-%d") #Datum in der Form YYYY-MM-DD
        except Exception:
            continue 
    return None 

df["Datum_standardisiert"] = df.apply(lambda row: parse_date(row["Datum"], row["Load-Date"], row["Text"]), axis=1)

df.to_csv("artikel_uebersicht_text_datum.csv", index=False, sep = ";", encoding="utf-8", quotechar = '"')

total = len(df)
parsed = df["Datum_standardisiert"].notna().sum()

print(f"Datum wurde vereinheitlicht und gespeichert. {parsed}/{total} Angaben vereinheitlicht")

/opt/anaconda3/lib/python3.12/site-packages/dateutil/parser/_parser.py:1207: UnknownTimezoneWarning: tzname EST identified but not understood.  Pass `tzinfos` argument in order to correctly return a timezone-aware datetime.  In a future version, this will raise an exception.
  warnings.warn("tzname {tzname} identified but not understood.  "


Datum wurde vereinheitlicht und gespeichert. 13557/13661 Angaben vereinheitlicht


### Gefilterte und angepasste Texte als txt-Dateien neu Speichern 

In [11]:
import os
import shutil

original_ordner = ordner_pfad #ggf. in erstem Schritt anpassen, wenn Fehler hier Auftritt
ziel_ordner = "/Users/lennertbusse/Documents/01Universität/DH/Projekt/texte_gefiltert"
#hier muss dafür gesorgt werden dass die processten Texte gespeichert werden

for filename in filtered_filenames:
    quelle = os.path.join(original_ordner, filename)
    ziel = os.path.join(ziel_ordner, filename)
    try:
        shutil.copy(quelle, ziel)
    except FileNotFoundError:
        print(f"Error, Datei nicht gefunden.")
print(f"{len(filtered_texts)} = {len(filtered_filenames)} Dateien wurden gespeichert.")

10938 = 10938 Dateien wurden gespeichert.


### CSV säubern und optimieren

In [12]:
import pandas as pd

def csv_saubern(pfad):
    df = pd.read_csv(pfad, sep = ";", encoding = "utf-8", quotechar = '"')

    leere_zeilen = df["Text"].isna() | (df["Text"].str.strip() == "")
    anzahl_vorher = len(df)

    df = df[~leere_zeilen].copy()
    anzahl_danach =len(df)

    df.to_csv("artikel_korpus_fin.csv", index=False, sep = ";", encoding = "utf-8", quotechar = '"')

    df = df.drop(columns=["Zeitung", "Datum", "Load-Date"], errors = "ignore")

    if "Datum_standardisiert" in df.columns:
        spalte = df.pop("Datum_standardisiert")
        df.insert(2, "Datum_standardisiert", spalte)

    neue_datei = pfad.replace(".csv", "_bereinigt.csv")
    df.to_csv(neue_datei, index=False, sep = ";", encoding = "utf-8", quotechar = '"')
    
    print(f"{anzahl_vorher - anzahl_danach} von {anzahl_vorher} Zeilen wurden aus der CSV gelöscht. {anzahl_danach} Artikel verbleiben in der Datei {neue_datei}.")

csv_saubern("artikel_uebersicht_text_datum.csv")

2723 von 13661 Zeilen wurden aus der CSV gelöscht. 10938 Artikel verbleiben in der Datei artikel_uebersicht_text_datum_bereinigt.csv.
